In [1]:
import json
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
from tqdm import tqdm

import torch
import open_clip

PARQUET_PATH = Path("/Users/zhasik/Desktop/krisha/data/index/index.parquet")
PROJECT_ROOT = Path("/Users/zhasik/Desktop/krisha")
OUT_DIR = PROJECT_ROOT / "data" / "processed"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("torch:", torch.__version__)
print("device:", "cpu")


torch: 2.9.1
device: cpu


In [2]:
df = pd.read_parquet(PARQUET_PATH)

def ensure_list(x):
    if isinstance(x, (list, tuple)):
        return list(x)
    if isinstance(x, np.ndarray):
        return x.tolist()
    if x is None:
        return []
    if isinstance(x, str):
        s = x.strip()
        if not s:
            return []
        if s.startswith("[") and s.endswith("]"):
            try:
                return json.loads(s)
            except Exception:
                pass
        return [p.strip().strip('"').strip("'") for p in s.split(",") if p.strip()]
    return []

df["image_paths_list"] = df["image_paths"].apply(ensure_list)
df["ad_id_str"] = df["ad_id"].astype(str)

df[["ad_id_str", "image_paths_list"]].head(2)


,ad_id_str,image_paths_list
0,1002526087,"[data/images/1002526087/01.jpg, data/images/10..."
1,1005774259,"[data/images/1005774259/01.jpg, data/images/10..."


In [3]:
def resolve_path(p: str) -> Path:
    p = str(p)
    ap = Path(p)
    if ap.is_absolute():
        return ap
    return (PROJECT_ROOT / p).resolve()

# quick sanity check
sample = df.iloc[0]["image_paths_list"][:2]
print(sample)
print([resolve_path(x).exists() for x in sample])
print([resolve_path(x) for x in sample])


['data/images/1002526087/01.jpg', 'data/images/1002526087/02.jpg']
[True, True]
[PosixPath('/Users/zhasik/Desktop/krisha/data/images/1002526087/01.jpg'), PosixPath('/Users/zhasik/Desktop/krisha/data/images/1002526087/02.jpg')]


In [4]:
device = torch.device("cpu")

model_name = "ViT-B-32"
pretrained = "laion2b_s34b_b79k"  # популярные веса для open_clip

model, _, preprocess = open_clip.create_model_and_transforms(
    model_name, pretrained=pretrained, device=device
)
model.eval()

# размер эмбеддинга
with torch.no_grad():
    dummy = torch.zeros(1, 3, 224, 224, device=device)
    z = model.encode_image(dummy)
    emb_dim = int(z.shape[-1])
print("embedding dim:", emb_dim)


open_clip_model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

embedding dim: 512


In [5]:
def load_image(path: Path):
    # robust loading
    img = Image.open(path).convert("RGB")
    return preprocess(img)

@torch.no_grad()
def encode_batch(image_tensors: torch.Tensor) -> np.ndarray:
    # image_tensors: [B,3,H,W]
    feats = model.encode_image(image_tensors)
    feats = torch.nn.functional.normalize(feats, dim=-1)
    return feats.cpu().numpy()


In [6]:
BATCH_SIZE = 32  # на CPU нормально 16-64; если тормозит — поставь 16

ad_ids = []
ad_embs = []

# можно для скорости заранее сделать плоский список (ad_id, img_path), но проще так:
for ad_id, paths in tqdm(zip(df["ad_id_str"].values, df["image_paths_list"].values), total=len(df)):
    tensors = []
    for p in paths[:7]:
        fp = resolve_path(p)
        if fp.exists():
            try:
                tensors.append(load_image(fp))
            except Exception:
                pass

    if len(tensors) == 0:
        # на всякий: если не загрузилось ни одно фото
        emb = np.zeros((emb_dim,), dtype=np.float32)
    else:
        # батчим внутри объявления (обычно 7 штук — маленький батч)
        imgs = torch.stack(tensors, dim=0).to(device)
        feats = encode_batch(imgs)             # [k, dim]
        emb = feats.mean(axis=0).astype(np.float32)
        emb = emb / (np.linalg.norm(emb) + 1e-12)

    ad_ids.append(ad_id)
    ad_embs.append(emb)

ad_ids = np.array(ad_ids)
ad_embs = np.stack(ad_embs, axis=0)

print("ad_ids:", ad_ids.shape, "embs:", ad_embs.shape)


100%|██████████| 1885/1885 [04:35<00:00,  6.85it/s]

ad_ids: (1885,) embs: (1885, 512)


In [7]:
np.save(OUT_DIR / "clip_vitb32_ad_ids.npy", ad_ids)
np.save(OUT_DIR / "clip_vitb32_ad_emb.npy", ad_embs)

print("Saved to:", OUT_DIR)


Saved to: /Users/zhasik/Desktop/krisha/data/processed


In [8]:
# sanity: близость у одинаковых объявлений (нет), но посмотрим статистику
norms = np.linalg.norm(ad_embs, axis=1)
print("norm min/mean/max:", norms.min(), norms.mean(), norms.max())


norm min/mean/max: 0.9999999 1.0 1.0000001
